# Imports, Helpers, Parameters

### Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from textwrap import wrap
import numpy as np
import os
import re
import matplotlib.font_manager as fm




### Parameters

In [2]:
main_color = "#3a5f83"

font_dir = r"C:\Users\teddy\Downloads\OAIPR\Technical\AEI Data Other\Lato"

# Loop through every file in the folder
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith(".ttf") and "lato" in font_file.lower():
        font_path = os.path.join(font_dir, font_file)
        fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.weight"] = "normal"

palette = sns.color_palette("colorblind")
# Put once at the TOP of your notebook/script (or just tweak this line)
sns.set_context("notebook", font_scale=1.0)  # was 1.2; smaller = less crowded

chart_size = (17, 9)

### Load Data

In [3]:
# Variants

# variant_name = "mask_directive_gt_0.1"
# variant_name = "mask_directive_gt_0.25"
# variant_name = "mask_directive_gt_0.5"
# variant_name = "mask_feedback_directive_gt_0.5"
# variant_name = "mask_feedback_directive_above_median"
# variant_name = "mask_feedback_directive_task_iter_gt_0.67"
# variant_name = "score_1_1_0_0_0"
# variant_name = "score_.8_1_.5_.2_.2"
variant_name = "score_1_1_.5_.5_.5"

base_dir = "../data/v2_auto_aug_variations"
automation_tasks_imputed = pd.read_csv(f"{base_dir}/automation_tasks_imputed_v2_{variant_name}.csv")

outdir = f"../outputs/charts_for_sharing/executive_summary_oct_2025/v2_{variant_name}"
os.makedirs(outdir, exist_ok=True)



# Base

# automation_tasks_imputed = pd.read_csv("../data/automation_tasks_imputed_v2.csv")

# outdir = "../outputs/charts_for_sharing/executive_summary_oct_2025/v2_base"
# os.makedirs(outdir, exist_ok=True)

# Original Charts

## Charts

### Workers Automated by Major Occupational Category National

In [150]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_nat']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Affected by Major Occupational Category (National)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Maj Occ Cat Tasks Affected)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Affected by Major Occupational Category (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)
grouped

,major_occ_category,people_automated_nat,ai_task_comp_nat,task_comp_nat,pct_automated_nat
0,Office and Administrative Support Occupations,4.976486e+06,2.776610e+08,9.833033e+08,28.237574
1,Educational Instruction and Library Occupations,2.502229e+06,1.529182e+08,4.210868e+08,36.315120
2,Business and Financial Operations Occupations,2.346430e+06,4.004532e+07,3.149397e+08,12.715235
3,Sales and Related Occupations,2.187021e+06,1.706366e+08,1.031300e+09,16.545785
4,Management Occupations,1.598971e+06,2.762333e+07,3.103117e+08,8.901802
5,Healthcare Support Occupations,1.186846e+06,4.285566e+07,3.963231e+08,10.813313
6,Computer and Mathematical Occupations,1.145806e+06,2.630777e+07,1.378586e+08,19.083157
7,Food Preparation and Serving Related Occupations,9.534339e+05,9.952193e+07,2.057117e+09,4.837931
8,Healthcare Practitioners and Technical Occupat...,7.541197e+05,6.172948e+07,1.796015e+09,3.437025
9,"Arts, Design, Entertainment, Sports, and Media...",5.096093e+05,1.731006e+07,8.370822e+07,20.679041


### Workers Automated by Major Occupational Category Utah

In [151]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_ut']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.02)  # Add 2% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Affected by Major Occupational Category (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Maj Occ Cat Tasks Affected)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Affected by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation Utah

In [152]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_ut', y='title_with_category', 
            color=main_color, ax=ax)

for i, (idx, row) in enumerate(top_15.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_nat']
    ax.text(value, i, f' {value:,.0f} ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.05)  # Add 5% padding on right side

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Affected Occupations by Workers (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Occ Tasks Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Most Affected Occupations by Workers (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Economic Value Generated by Major Occupational Category Utah

In [153]:
# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "eco_value_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by economic value and get all categories
grouped = grouped.sort_values("eco_value_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='eco_value_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add labels with dollar amount (in billions) and percentage
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['eco_value_ut'] / 1e9  # convert to billions
    pct = row['pct_automated_ut']
    ax.text(row['eco_value_ut'], i, f' ${value:,.1f}B ({pct:.1f}%)', 
            va='center', ha='left', fontsize=9)
    
# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.1)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Wages Generated by Major Occupational Category (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Wages Generated (% Maj Occ Cat Automated)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis in billions
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e9:,.1f}B"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Wages Generated by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Total Utah Economic Value

In [154]:
# Load the data
eco_2025 = pd.read_csv('../data/ratings_eco_2025.csv')

# Get one row per occupation (since employment/wage are same across all tasks for an occupation)
occupation_level = eco_2025.drop_duplicates(subset='title').copy()

# Compute total payroll (wage * employment)
occupation_level['total_payroll_ut'] = occupation_level['tot_emp_ut'] * occupation_level['a_med_ut']

# Sum total employment and total payroll across all occupations
total_payroll_ut = occupation_level['total_payroll_ut'].sum()

# Optional: sort by total payroll to see which occupations contribute most
occupation_level_sorted = occupation_level.sort_values('total_payroll_ut', ascending=False)

# Display results
print(f"Total Utah Payroll (Employment × Wage): ${total_payroll_ut:,.0f}")

# Optional preview
occupation_level_sorted[['title', 'tot_emp_ut', 'a_med_ut', 'total_payroll_ut']].head()


Total Utah Payroll (Employment × Wage): $133,941,368,538


,title,tot_emp_ut,a_med_ut,total_payroll_ut
49,General and Operations Managers,45910.0,91230.0,4.188369e+09
8043,Registered Nurses,25780.0,82270.0,2.120921e+09
7992,Advanced Practice Psychiatric Nurses,25780.0,82270.0,2.120921e+09
8008,Clinical Nurse Specialists,25780.0,82270.0,2.120921e+09
8045,Acute Care Nurses,25780.0,82270.0,2.120921e+09


# Simplified Charts

## Load Data

In [7]:
automation_tasks_imputed = pd.read_csv("../data/automation_tasks_imputed_v1.csv")
automation_tasks_imputed['major_occ_category'] = automation_tasks_imputed['major_occ_category'].str.replace(' Occupations', '', regex=False)

## Charts

### Workers Automated by Major Occupational Category National

In [34]:
# Base font size variable
font_size = 19

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by people automated and get TOP 6
grouped = grouped.sort_values("people_automated_nat", ascending=False).head(6).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars with proportional font
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_nat']
    pct = row['pct_automated_nat']
    ax.text(
        value * 0.98, i, f' {value:,.0f} ({pct:.1f}%)',
        va='center', ha='right', color="white",
        fontsize=font_size
    )

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.00)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=font_size)

plt.subplots_adjust(left=0.38)

# Titles and axis labels (scaled proportionally)
ax.set_title(
    "Top 6 Workers Affected by Major Occupational Category (National)",
    fontsize=font_size * 1.1, fontweight='bold', pad=20
)
ax.set_xlabel(
    "Number of Workers Affected (% Maj Occ Cat Tasks Affected)",
    fontsize=font_size * 0.84, labelpad=10
)
ax.set_ylabel(
    "Major Occupational Category",
    fontsize=font_size * 0.84, labelpad=10
)

# X-axis tick labels
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis='x', labelsize=font_size)

# Grid
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save output
filename = "Top 6 Workers Affected by Major Occupational Category (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)


### Workers Automated by Major Occupational Category Utah

In [42]:
# Base font size variable
font_size = 19

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).head(6).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Add value labels at end of bars
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_ut']
    ax.text(
        value * 0.98, i, f' {value:,.0f} ({pct:.1f}%)',
        va='center', ha='right', color="white",
        fontsize=font_size
    )

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.00)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=font_size)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Top 6 Workers Affected by Major Occupational Category (Utah)", 
             fontsize=font_size * 1.1, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Affected (% Maj Occ Cat Tasks Affected)", fontsize=font_size * 0.84, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=font_size * 0.84, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis='x', labelsize=font_size)

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Top 6 Workers Affected by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 5 Workers Automated by Occupation Utah

In [36]:
# Base font size variable
font_size = 19

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 5 most automated occupations
top_5 = subset.nlargest(5, 'people_automated_ut').copy()

# Create combined label with title and major category
top_5['title_with_category'] = top_5.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(
    data=top_5,
    x='people_automated_ut',
    y='title_with_category',
    color=main_color,
    ax=ax
)

# Add value labels on bars (scaled font size)
for i, (idx, row) in enumerate(top_5.iterrows()):
    value = row['people_automated_ut']
    pct = row['pct_automated_nat']
    ax.text(
        value * 0.98, i,
        f'{value:,.0f} ({pct:.1f}%)',
        va='center', ha='right',
        color='white',
        fontsize=font_size
    )

# Extend x-axis limit slightly for spacing
ax.set_xlim(right=ax.get_xlim()[1] * 1.00)

# Wrap y labels (scaled font)
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=30)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=font_size)
plt.subplots_adjust(left=0.42)

# Professional titles and labels (scaled from base)
ax.set_title(
    "Top 5 Most Affected Occupations by Workers (Utah)",
    fontsize=font_size * 1.1,
    fontweight='bold',
    pad=20
)
ax.set_xlabel(
    "Number of Workers Affected (% Occ Tasks Automated)",
    fontsize=font_size * 0.84,
    labelpad=10
)
ax.set_ylabel(
    "Occupation [Major Occupational Category]",
    fontsize=font_size * 0.84,
    labelpad=10
)

# Format x-axis tick labels
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.tick_params(axis='x', labelsize=font_size)

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Top 5 Most Affected Occupations by Workers (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Economic Value Generated by Major Occupational Category Utah


In [40]:
# Base font size variable
font_size = 19

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "eco_value_ut": "sum",
    "ai_task_comp_ut": "sum",
    "task_comp_ut": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_ut"] = (grouped["ai_task_comp_ut"] / grouped["task_comp_ut"]) * 100

# Sort by economic value and get TOP 5
grouped = grouped.sort_values("eco_value_ut", ascending=False).head(6).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(
    data=grouped,
    x='eco_value_ut',
    y='major_occ_category',
    color=main_color,
    ax=ax
)

# Add labels with dollar amount (in billions) and percentage
for i, (idx, row) in enumerate(grouped.iterrows()):
    value = row['eco_value_ut'] / 1e9  # convert to billions
    pct = row['pct_automated_ut']
    ax.text(
        row['eco_value_ut'] * 0.98, i,
        f'${value:,.1f}B ({pct:.1f}%)',
        va='center', ha='right',
        color='white',
        fontsize=font_size
    )

# Extend x-axis limit slightly to prevent label overlap with border
ax.set_xlim(right=ax.get_xlim()[1] * 1.00)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=font_size)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title(
    "Top 6 Wages Generated by Major Occupational Category (Utah)",
    fontsize=font_size * 1.1,
    fontweight='bold',
    pad=20
)
ax.set_xlabel(
    "Wages Generated (% Maj Occ Cat Automated)",
    fontsize=font_size * 0.84,
    labelpad=10
)
ax.set_ylabel(
    "Major Occupational Category",
    fontsize=font_size * 0.84,
    labelpad=10
)

# Format x-axis in billions
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x/1e9:,.1f}B"))
ax.tick_params(axis='x', labelsize=font_size)

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

plt.tight_layout()

# Save with descriptive filename
filename = "Top 6 Wages Generated by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)


# Auto Aug Variant Comparisons

## Create Data Frames

### Create General Summary Data Frame

In [4]:
base_path = "../data"
base_file = "automation_tasks_imputed_v2.csv"
variant_path = os.path.join(base_path, "v2_auto_aug_variations")

variants = [
    "automation_tasks_imputed_v2_mask_directive_gt_0.1.csv",
    "automation_tasks_imputed_v2_mask_directive_gt_0.25.csv",
    "automation_tasks_imputed_v2_mask_directive_gt_0.5.csv",
    "automation_tasks_imputed_v2_mask_feedback_directive_gt_0.5.csv",
    "automation_tasks_imputed_v2_mask_feedback_directive_above_median.csv",
    "automation_tasks_imputed_v2_mask_feedback_directive_task_iter_gt_0.67.csv",
    "automation_tasks_imputed_v2_score_1_1_0_0_0.csv",
    "automation_tasks_imputed_v2_score_.8_1_.5_.2_.2.csv",
    "automation_tasks_imputed_v2_score_1_1_.5_.5_.5.csv",
]



def aggregate(df):
    """Aggregate to major occupation category level for people and wages."""
    df = df[df["freq_sum_ai"] <= df["freq_sum_eco"]].copy()
    grouped = df.groupby("major_occ_category").agg({
        "people_automated_ut": "sum",
        "eco_value_ut": "sum",       # add wages
        "ai_task_comp_ut": "sum",
        "task_comp_ut": "sum"
    }).reset_index()
    grouped["pct_automated_ut"] = (
        grouped["ai_task_comp_ut"] / grouped["task_comp_ut"] * 100
    )
    return grouped

# Load baseline 
base = pd.read_csv(os.path.join(base_path, base_file))
base_grouped = aggregate(base).rename(columns={
    "people_automated_ut": "people_auto_base",
    "eco_value_ut": "eco_base",
    "pct_automated_ut": "pct_auto_base"
})

# Compare each variant
results = []

for fname in variants:
    path = os.path.join(variant_path, fname)
    df = pd.read_csv(path)
    variant_grouped = aggregate(df).rename(columns={
        "people_automated_ut": "people_auto_variant",
        "eco_value_ut": "eco_variant",
        "pct_automated_ut": "pct_auto_variant"
    })

    comp = base_grouped.merge(variant_grouped, on="major_occ_category", how="outer")
    comp["variant"] = fname

    # Worker deltas
    comp["Δ people"] = comp["people_auto_variant"] - comp["people_auto_base"]
    comp["Δ pct"] = comp["pct_auto_variant"] - comp["pct_auto_base"]
    comp["% change in people"] = comp["Δ people"] / comp["people_auto_base"] * 100
    comp["% change in pct"] = comp["Δ pct"] / comp["pct_auto_base"] * 100

    # Wage deltas
    comp["Δ eco_value"] = comp["eco_variant"] - comp["eco_base"]
    comp["% change in eco_value"] = comp["Δ eco_value"] / comp["eco_base"] * 100

    results.append(comp)

# ---------- Combine ----------
comparison_table = pd.concat(results, ignore_index=True)

# Order columns for clarity
comparison_table = comparison_table[
    [
        "variant",
        "major_occ_category",
        "people_auto_base", "people_auto_variant", "Δ people", "% change in people",
        "eco_base", "eco_variant", "Δ eco_value", "% change in eco_value",
        "pct_auto_base", "pct_auto_variant", "Δ pct", "% change in pct",
    ]
]

# ---------- Save ----------
comparison_table.to_csv(
    os.path.join(variant_path, "automation_variant_comparison_summary.csv"),
    index=False
)


### Create Data Frame For Tables

In [5]:
# ---------- Load data ----------
comparison_table = pd.read_csv(
    os.path.join("../data/v2_auto_aug_variations", "automation_variant_comparison_summary.csv")
)

# ---------- Focus categories ----------
focus_categories = [
    "Office and Administrative Support Occupations",
    "Sales and Related Occupations",
    "Educational Instruction and Library Occupations",
    "Business and Financial Operations Occupations",
    "Management Occupations",
    "Computer and Mathematical Occupations",
]

# Filter + keep order
filtered = (
    comparison_table[comparison_table["major_occ_category"].isin(focus_categories)]
    .copy()
)
filtered["major_occ_category"] = pd.Categorical(
    filtered["major_occ_category"], categories=focus_categories, ordered=True
)

# ---------- Create baseline summary ----------
baseline = (
    filtered[["major_occ_category", "people_auto_base", "eco_base", "pct_auto_base"]]
    .drop_duplicates(subset=["major_occ_category"])
    .rename(columns={
        "people_auto_base": "Base: People",
        "eco_base": "Base: Wages",
        "pct_auto_base": "Base: % Automated"
    })
)

# ---------- Reshape variant deltas ----------
variant_cols = [
    "Δ people", "% change in people",
    "Δ eco_value", "% change in eco_value",
    "Δ pct", "% change in pct"
]

# Pivot so each variant’s deltas become grouped columns
variants_wide = (
    filtered[["variant", "major_occ_category"] + variant_cols]
    .set_index(["major_occ_category", "variant"])
    .unstack(level="variant")
)

# Flatten multiindex columns
variants_wide.columns = [
    f"{col[1].replace('.csv','').replace('automation_tasks_imputed_v2_','').replace('_',' ').title()} | {col[0]}"
    for col in variants_wide.columns
]

# ---------- Combine baseline and deltas ----------
final = baseline.merge(variants_wide, on="major_occ_category", how="left")

# ---------- Formatting ----------
final = final.round(2)

# ---------- Display or save ----------
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
final

# Optional: save to CSV
final.to_csv(
    os.path.join("../data/v2_auto_aug_variations", "automation_variant_summary_matrix.csv"),
    index=False
)


## Create Tables

### Create Table With All Values Comp

In [56]:
file_path = os.path.join("../data/v2_auto_aug_variations", "automation_variant_summary_matrix.csv")
df = pd.read_csv(file_path)

# Identify which columns should be styled
pct_cols = [c for c in df.columns if "% change" in c or c.endswith("%")]
count_cols = [c for c in df.columns if "Δ" in c and c not in pct_cols]

# ---------- Define color functions ----------
def red_gradient(val):
    """Deeper red for larger % magnitude."""
    if pd.isna(val):
        return ""
    intensity = min(abs(val) / 50, 1)  # scale up to ~±50%
    return f"background-color: rgba(220, 53, 69, {0.15 + 0.5 * intensity});"

def blue_gradient(val):
    """Deeper blue for larger absolute count."""
    if pd.isna(val):
        return ""
    intensity = min(abs(val) / df[count_cols].abs().max().max(), 1)
    return f"background-color: rgba(0, 123, 255, {0.15 + 0.5 * intensity});"

# ---------- Apply formatting ----------
styled = (
    df.style
    .format(precision=2, thousands=",")
    .applymap(red_gradient, subset=pct_cols)
    .applymap(blue_gradient, subset=count_cols)
    .set_caption("Automation Variant Summary — Top Six Occupational Categories")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "16px"), ("font-weight", "bold"), ("text-align", "center"), ("color", "#333")]},
        {"selector": "th",
         "props": [("background-color", "#f2f2f2"), ("font-weight", "bold")]},
    ])
)

# ---------- Show in Jupyter ----------
styled

# # ---------- Optional: export as HTML ----------
# styled.to_html(os.path.join("../data/v2_auto_aug_variations", "automation_variant_summary_matrix_styled.html"))
# print("✅ Saved styled table to automation_variant_summary_matrix_styled.html")


C:\Users\teddy\AppData\Local\Temp\ipykernel_8244\3328961548.py:27: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(red_gradient, subset=pct_cols)
C:\Users\teddy\AppData\Local\Temp\ipykernel_8244\3328961548.py:28: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(blue_gradient, subset=count_cols)


,major_occ_category,Base: People,Base: Wages,Base: % Automated,Above Auto Median | Δ people,Combined Score Aug Half | Δ people,Only Feedback Directive | Δ people,Only Feedback Directive Task Iteration | Δ people,Score .8 1 .5 .2 .2 | Δ people,Score Filled .8 1 .5 .2 .2 | Δ people,Score Filled 1 1 .5 .5 .5 | Δ people,Above Auto Median | % change in people,Combined Score Aug Half | % change in people,Only Feedback Directive | % change in people,Only Feedback Directive Task Iteration | % change in people,Score .8 1 .5 .2 .2 | % change in people,Score Filled .8 1 .5 .2 .2 | % change in people,Score Filled 1 1 .5 .5 .5 | % change in people,Above Auto Median | Δ eco_value,Combined Score Aug Half | Δ eco_value,Only Feedback Directive | Δ eco_value,Only Feedback Directive Task Iteration | Δ eco_value,Score .8 1 .5 .2 .2 | Δ eco_value,Score Filled .8 1 .5 .2 .2 | Δ eco_value,Score Filled 1 1 .5 .5 .5 | Δ eco_value,Above Auto Median | % change in eco_value,Combined Score Aug Half | % change in eco_value,Only Feedback Directive | % change in eco_value,Only Feedback Directive Task Iteration | % change in eco_value,Score .8 1 .5 .2 .2 | % change in eco_value,Score Filled .8 1 .5 .2 .2 | % change in eco_value,Score Filled 1 1 .5 .5 .5 | % change in eco_value,Above Auto Median | Δ pct,Combined Score Aug Half | Δ pct,Only Feedback Directive | Δ pct,Only Feedback Directive Task Iteration | Δ pct,Score .8 1 .5 .2 .2 | Δ pct,Score Filled .8 1 .5 .2 .2 | Δ pct,Score Filled 1 1 .5 .5 .5 | Δ pct,Above Auto Median | % change in pct,Combined Score Aug Half | % change in pct,Only Feedback Directive | % change in pct,Only Feedback Directive Task Iteration | % change in pct,Score .8 1 .5 .2 .2 | % change in pct,Score Filled .8 1 .5 .2 .2 | % change in pct,Score Filled 1 1 .5 .5 .5 | % change in pct
0,Business and Financial Operations Occupations,"38,000.09","2,738,378,930.64",33.19,"-17,820.59","-24,537.01","-16,184.13","-14,244.87","-25,791.22","-17,984.18","-15,366.53",-46.90,-64.57,-42.59,-37.49,-67.87,-47.33,-40.44,"-1,318,233,505.68","-1,796,587,690.69","-1,213,287,552.86","-1,088,672,993.04","-1,880,212,540.56","-1,293,517,515.30","-1,108,312,638.47",-48.14,-65.61,-44.31,-39.76,-68.66,-47.24,-40.47,-16.71,-21.62,-15.08,-13.02,-22.67,-15.67,-13.28,-50.35,-65.15,-45.42,-39.22,-68.29,-47.20,-40.02
1,Computer and Mathematical Occupations,"21,853.63","1,955,841,096.00",60.90,"-5,118.42","-10,464.61","-4,267.51","-3,816.03","-13,022.86","-11,608.71","-8,820.91",-23.42,-47.88,-19.53,-17.46,-59.59,-53.12,-40.36,"-500,874,362.39","-940,733,391.08","-417,781,615.77","-379,655,912.89","-1,158,380,482.98","-1,019,907,767.99","-780,336,348.63",-25.61,-48.10,-21.36,-19.41,-59.23,-52.15,-39.90,-17.04,-30.76,-15.46,-14.20,-37.44,-31.88,-24.22,-27.97,-50.51,-25.39,-23.31,-61.48,-52.35,-39.77
2,Educational Instruction and Library Occupations,"35,899.39","2,437,442,494.84",48.86,"-9,805.30","-17,123.53","-6,891.85","-6,882.39","-19,405.86","-15,785.41","-13,029.80",-27.31,-47.70,-19.20,-19.17,-54.06,-43.97,-36.30,"-740,622,227.79","-1,177,534,432.61","-487,386,313.11","-486,921,053.47","-1,335,938,342.89","-1,078,381,785.37","-886,121,278.59",-30.39,-48.31,-20.00,-19.98,-54.81,-44.24,-36.35,-9.53,-21.45,-7.17,-7.17,-24.31,-20.45,-17.11,-19.51,-43.90,-14.68,-14.68,-49.75,-41.85,-35.02
3,Management Occupations,"28,860.73","3,452,431,687.94",22.39,"-16,531.38","-19,412.22","-16,378.86","-9,841.74","-20,414.42","-14,826.39","-13,022.95",-57.28,-67.26,-56.75,-34.10,-70.73,-51.37,-45.12,"-2,004,207,491.00","-2,361,249,552.81","-1,981,440,341.13","-1,304,249,902.74","-2,486,374,893.42","-1,789,256,887.95","-1,567,232,599.02",-58.05,-68.39,-57.39,-37.78,-72.02,-51.83,-45.40,-13.01,-15.46,-12.83,-9.57,-16.38,-11.62,-10.07,-58.08,-69.05,-57.30,-42.73,-73.16,-51.87,-44.98
4,Office and Administrative Support Occupations,"86,903.13","3,839,752,862.48",39.26,"-42,858.67","-54,847.58","-27,071.69","-24,986.16","-59,637.92","-43,322.39","-36,394.45",-49.32,-63.11,-31.15,-28.75,-68.

### Create Table With Column Values Comp

In [7]:
# ---------- Load ----------
file_path = os.path.join("../data/v2_auto_aug_variations", "automation_variant_summary_matrix.csv")
df = pd.read_csv(file_path)

# Identify which columns should be styled
pct_cols = [c for c in df.columns if "% change" in c or c.endswith("%")]
count_cols = [c for c in df.columns if "Δ" in c and c not in pct_cols]

# ---------- Define color functions ----------
def colwise_red_gradient(series):
    """Deeper red for larger % magnitude (per column)."""
    max_val = series.abs().max()
    return [
        f"background-color: rgba(220, 53, 69, {0.15 + 0.5 * (abs(v)/max_val)})"
        if not pd.isna(v) else "" for v in series
    ]

def colwise_blue_gradient(series):
    """Deeper blue for larger absolute counts (per column)."""
    max_val = series.abs().max()
    return [
        f"background-color: rgba(0, 123, 255, {0.15 + 0.5 * (abs(v)/max_val)})"
        if not pd.isna(v) else "" for v in series
    ]

# ---------- Apply formatting ----------
styled = (
    df.style
    .format(precision=2, thousands=",")
    .apply(colwise_red_gradient, subset=pct_cols, axis=0)
    .apply(colwise_blue_gradient, subset=count_cols, axis=0)
    .set_caption("Automation Variant Summary — Top Six Occupational Categories")
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "16px"), ("font-weight", "bold"),
                   ("text-align", "center"), ("color", "#333")]},
        {"selector": "th",
         "props": [("background-color", "#f2f2f2"),
                   ("font-weight", "bold")]},
    ])
)

# ---------- Show in Jupyter ----------
styled

# # ---------- Optional: export as HTML ----------
# styled.to_html(os.path.join("../data/v2_auto_aug_variations", "automation_variant_summary_matrix_styled.html"))
# print("✅ Saved styled table to automation_variant_summary_matrix_styled.html")


,major_occ_category,Base: People,Base: Wages,Base: % Automated,Mask Directive Gt 0.1 | Δ people,Mask Directive Gt 0.25 | Δ people,Mask Directive Gt 0.5 | Δ people,Mask Feedback Directive Above Median | Δ people,Mask Feedback Directive Gt 0.5 | Δ people,Mask Feedback Directive Task Iter Gt 0.67 | Δ people,Score .8 1 .5 .2 .2 | Δ people,Score 1 1 .5 .5 .5 | Δ people,Score 1 1 0 0 0 | Δ people,Mask Directive Gt 0.1 | % change in people,Mask Directive Gt 0.25 | % change in people,Mask Directive Gt 0.5 | % change in people,Mask Feedback Directive Above Median | % change in people,Mask Feedback Directive Gt 0.5 | % change in people,Mask Feedback Directive Task Iter Gt 0.67 | % change in people,Score .8 1 .5 .2 .2 | % change in people,Score 1 1 .5 .5 .5 | % change in people,Score 1 1 0 0 0 | % change in people,Mask Directive Gt 0.1 | Δ eco_value,Mask Directive Gt 0.25 | Δ eco_value,Mask Directive Gt 0.5 | Δ eco_value,Mask Feedback Directive Above Median | Δ eco_value,Mask Feedback Directive Gt 0.5 | Δ eco_value,Mask Feedback Directive Task Iter Gt 0.67 | Δ eco_value,Score .8 1 .5 .2 .2 | Δ eco_value,Score 1 1 .5 .5 .5 | Δ eco_value,Score 1 1 0 0 0 | Δ eco_value,Mask Directive Gt 0.1 | % change in eco_value,Mask Directive Gt 0.25 | % change in eco_value,Mask Directive Gt 0.5 | % change in eco_value,Mask Feedback Directive Above Median | % change in eco_value,Mask Feedback Directive Gt 0.5 | % change in eco_value,Mask Feedback Directive Task Iter Gt 0.67 | % change in eco_value,Score .8 1 .5 .2 .2 | % change in eco_value,Score 1 1 .5 .5 .5 | % change in eco_value,Score 1 1 0 0 0 | % change in eco_value,Mask Directive Gt 0.1 | Δ pct,Mask Directive Gt 0.25 | Δ pct,Mask Directive Gt 0.5 | Δ pct,Mask Feedback Directive Above Median | Δ pct,Mask Feedback Directive Gt 0.5 | Δ pct,Mask Feedback Directive Task Iter Gt 0.67 | Δ pct,Score .8 1 .5 .2 .2 | Δ pct,Score 1 1 .5 .5 .5 | Δ pct,Score 1 1 0 0 0 | Δ pct,Mask Directive Gt 0.1 | % change in pct,Mask Directive Gt 0.25 | % change in pct,Mask Directive Gt 0.5 | % change in pct,Mask Feedback Directive Above Median | % change in pct,Mask Feedback Directive Gt 0.5 | % change in pct,Mask Feedback Directive Task Iter Gt 0.67 | % change in pct,Score .8 1 .5 .2 .2 | % change in pct,Score 1 1 .5 .5 .5 | % change in pct,Score 1 1 0 0 0 | % change in pct
0,Business and Financial Operations Occupations,"38,000.09","2,738,378,930.64",33.19,"-1,550.59","-4,260.04","-28,339.56","-10,960.21","-27,282.91","-7,475.59","-13,117.65","-10,855.65","-21,711.30",-4.08,-11.21,-74.58,-28.84,-71.80,-19.67,-34.52,-28.57,-57.13,"-95,792,595.53","-276,497,631.18","-2,077,395,338.79","-750,903,307.37","-2,000,082,293.16","-495,507,857.03","-934,854,627.85","-781,604,323.37","-1,563,208,646.74",-3.50,-10.10,-75.86,-27.42,-73.04,-18.09,-34.14,-28.54,-57.09,-1.92,-4.77,-23.63,-8.87,-19.71,-7.38,-11.38,-9.24,-18.47,-5.78,-14.37,-71.20,-26.73,-59.38,-22.23,-34.29,-27.83,-55.66
1,Computer and Mathematical Occupations,"21,853.63","1,955,841,096.00",60.90,"-3,952.78","-10,462.37","-20,847.45","-10,795.64","-16,891.86","-14,456.49","-10,237.30","-6,835.70","-13,671.40",-18.09,-47.87,-95.40,-49.40,-77.30,-66.15,-46.84,-31.28,-62.56,"-326,415,180.07","-933,943,494.18","-1,843,162,308.85","-970,299,739.48","-1,545,355,337.33","-1,210,021,505.37","-904,582,642.54","-614,286,160.70","-1,228,572,321.40",-16.69,-47.75,-94.24,-49.61,-79.01,-61.87,-46.25,-31.41,-62.82,-10.01,-29.97,-58.63,-30.19,-48.00,-38.42,-28.02,-18.94,-37.88,-16.44,-49.22,-96.28,-49.58,-78.82,-63.08,-46.01,-31.10,-62.21
2,Educational Instruction and Library Occupations,"35,899.39","2,437,442,494.84",48.86,-796.17,"-5,779.50","-20,561.37","-11,440.97","-20,439.94","-15,061.58","-12,651.60","-9,508.18","-19,016.36",-2.22,-16.10,-57.27,-31.87,-56.94,-41.95,-35.24,-26.49,-52.97,"-65,552,982.57","-469,564,844.19","-1,427,451,376.97","-803,809,897.41","-1,420,990,126.10","-978,629,205.92","-876,804,191.03","-658,474,194.35","-1,316,948,388.70",-2.69,-19.26,-58.56,-32.98,-58.